# Q‑Criterion Visualisation for Surface‑Water CFD

This mini‑notebook shows how to **compute and visualise the Q‑criterion** for a 3‑D velocity field stored in a VTK/VTP file (e.g. output from *OpenFOAM*, *Delft3D‑FM*, *TELEMAC‑3D*, *FLOW‑3D*, etc).

**What you’ll get**

* A new point‑data array `Q` added to the mesh  
* An interactive iso‑surface view with a slider for the threshold  
* Optionally, a VTP file you can open directly in ParaView

**Requirements**

```bash
pip install pyvista numpy
```

(You can skip if those packages are already available.)


In [ ]:
# If this is a brand‑new environment, uncomment the next line
# !pip install pyvista numpy

import numpy as np
import pyvista as pv


In [ ]:
# ---------- user parameters ----------
filename       = 'my_case.vtp'  # path to your VTK/VTP file
velocity_name  = 'U'            # name of the velocity vector array
iso_level      = None           # None ➜ 1×RMS(Q); or specify a positive value
# --------------------------------------


In [ ]:
mesh = pv.read(filename)

if velocity_name not in mesh.point_data:
    raise KeyError(f"'{velocity_name}' not found in point data. "
                   f"Available arrays: {list(mesh.point_data.keys())}")

print(f"Loaded mesh with {mesh.n_points:,} points and {mesh.n_cells:,} cells")


In [ ]:
# Compute velocity gradient (uses vtkGradientFilter under the hood)
grad = mesh.compute_derivative(scalars=velocity_name, gradient=True)
G    = grad.point_data[f'{velocity_name}_gradient'].reshape(-1, 3, 3)  # (N,3,3)

# Symmetric (strain) and anti‑symmetric (rotation) parts
S      = 0.5 * (G + np.transpose(G, (0, 2, 1)))
Omega  = 0.5 * (G - np.transpose(G, (0, 2, 1)))

# Q‑criterion: 0.5*(|Omega|^2 – |S|^2)
Q = 0.5 * (np.sum(Omega**2, axis=(1, 2)) - np.sum(S**2, axis=(1, 2)))

mesh.point_data['Q'] = Q
print("Added 'Q' array to point data")


In [ ]:
if iso_level is None:
    iso_level = np.sqrt(np.mean(Q**2))  # 1×RMS
    print(f"Using iso‑level = {iso_level:.3e} (RMS(Q))")
else:
    print(f"Using user‑defined iso‑level = {iso_level}")


In [ ]:
# Precompute min/max slider range
slider_min = 0.1 * iso_level
slider_max = 5.0 * iso_level

# Helper for live updates
def update_threshold(value):
    plotter.clear()
    cont = mesh.contour(isosurfaces=[value], scalars='Q')
    plotter.add_mesh(cont, scalars='Q')
    plotter.add_axes()
    plotter.show_grid()
    plotter.add_text(f'Q = {value:.2e}', position='upper_left')
    plotter.render()

plotter = pv.Plotter()
plotter.add_slider_widget(
    callback=update_threshold,
    rng=[slider_min, slider_max],
    value=iso_level,
    title='Q iso‑level'
)

update_threshold(iso_level)  # initial draw
plotter.show()


In [ ]:
# Optional: write mesh with Q array for ParaView
# mesh.save('mesh_with_Q.vtp')
# print('Saved mesh_with_Q.vtp')
